# Multilingual Threat Intelligence Classifier

Run on Colab with a GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
!git clone https://github.com/Lucia-Navarro12/multilingual-threat-intel-classifier.git
%cd multilingual-threat-intel-classifier
!pip install -q -r requirements.txt

## 1. Data: English splits + Spanish machine translation

In [ ]:
!python src/data.py
!python src/translate.py

In [ ]:
import pandas as pd
en, es = pd.read_csv('data/en/test.csv'), pd.read_csv('data/es/test.csv')
for i in range(3):
    print('EN:', en.text[i]); print('ES:', es.text[i]); print()

## 2. Baseline: TF-IDF + logistic regression

Trained on English only, then on both languages. A bag of English words cannot read Spanish.

In [ ]:
!python src/baseline.py --train-langs en
!python src/baseline.py --train-langs en es

## 3. Fine-tuning multilingual BERT

First English only (zero-shot transfer to Spanish), then both languages.

In [ ]:
!python src/train.py --train-langs en

In [ ]:
!python src/train.py --train-langs en es

## 4. Results

In [ ]:
import json, glob
rows = []
for f in sorted(glob.glob('results/*.json')):
    for lang, m in json.load(open(f)).items():
        rows.append({'run': f.split('/')[-1][:-5], 'test_lang': lang, 'f1_micro': round(m['f1_micro'], 3), 'f1_macro': round(m['f1_macro'], 3)})
pd.DataFrame(rows)

## 5. Try it

In [ ]:
!python src/predict.py --model-dir models/bert-base-multilingual-cased_train-en-es/best \
  "Ransomware gang encrypts hospital servers and demands bitcoin" \
  "Filtrados los datos personales de dos millones de clientes de una aerolínea" \
  "Ataque DDoS deja sin servicio las webs de varios ministerios"